# Ordered Logistic Regression Results: FAIRⁿ² Dataset Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the FAIRⁿ² dataset using the `mlcroissant` library, following the [Croissant schema](https://mlcommons.org/croissant/). You will learn how to load the dataset metadata, review available record sets and fields, import the data for analysis, and perform some exploratory processing and visualization.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and discover the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets and their fields and corresponding `@id`s per the Croissant schema.

In [ ]:
# Print out record sets and their fields using @id references
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print()
if not record_sets:
    print("No record sets were declared in the dataset metadata.")

# For demonstration, try to iterate over the first record set (if exists)
if record_sets:
    first_record_set_id = record_sets[0].id
    print(f"\nSample record from record set @id: {first_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No records to show.")

## 3. Data Extraction
Load records from specific record sets using their `@id`s into Pandas DataFrames for analysis.

In [ ]:
# Collect all record set @ids
record_sets = dataset.record_sets
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
# For each record set, extract records into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set @id: {record_set_id}")
    print(f"Columns: {list(df.columns)}\n")

# Show head of first DataFrame (if any)
if dataframes:
    example_rs_id = record_set_ids[0]
    print(f"First few records from record set @id: {example_rs_id}")
    display(dataframes[example_rs_id].head())
else:
    print("No dataframes were created because no record sets were present.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric columns, and optionally group by a categorical field. All columns are referenced using their `@id`s.

In [ ]:
# If there are record sets and dataframes, perform EDA on the first one
if dataframes:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Identify numeric fields by inspecting DataFrame dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use the @id of a numeric column
        print(f"Numeric field selected for analysis: {numeric_field_id}")

        # Set a threshold value for filtering (example: 10, or use quantiles if needed)
        threshold = df[numeric_field_id].quantile(0.75) if df[numeric_field_id].nunique()>20 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt grouping by a non-numeric field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric group field found.")
    else:
        print("No numeric fields found in the sample record set. Skipping EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and the results of any grouping (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use same field and groupings as in EDA if available
if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping was done
    if 'group_field' in locals() and group_field:
        if 'grouped_df' in locals() and not grouped_df.empty:
            plt.figure(figsize=(10, 5))
            sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
            plt.title(f"Mean {numeric_field_id} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, inspect, and analyze the FAIRⁿ² dataset (Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya). We explored available record sets and fields, loaded records into Pandas DataFrames, performed basic EDA, and visualized some data distributions. All dataset entities and fields were referenced using their `@id`s as per the Croissant standard, ensuring clarity and reproducibility for downstream analysis.